<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px 30px; border-radius: 12px; margin-bottom: 10px;">
    <h1 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:2.2em; margin:0 0 8px 0;">
        🎓 Introdução ao Aprendizado de Máquina
    </h1>
    <h2 style="color:#a8d8ea; font-family:'Segoe UI', sans-serif; font-size:1.3em; margin:0 0 6px 0; font-weight:400;">
        Aula 20 — APIs e Deploy com Flask
    </h2>
    <h3 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:1.05em; margin:0 0 12px 0; font-weight:500;">
        🔬 Prática — O Oráculo do Titanic Vira uma API
    </h3>
    <p style="color:#ccc; font-family:'Segoe UI', sans-serif; font-size:0.9em; margin:0;">
        Prof. Felipe Amaral
    </p>
</div>
<div style="display:flex; gap:10px; margin-top:10px; flex-wrap:wrap;">
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">📚 FIAP</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🐍 Python 3</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🚢 Dataset Titanic</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🌐 Flask + API REST</span>
</div>

## Onde estamos?

| Aula | Tema | Ideia central |
|------|------|---------------|
| 02–03 | EDA e Preparação | Entender e limpar o Titanic |
| 04–09 | Modelos supervisionados | Reg. Linear, Reg. Logística, KNN, SVM, Árvore, Random Forest |
| 10 | K-Means | Encontrar grupos sem rótulos |
| **20** | **API com Flask** | **Tirar o modelo do notebook e colocar no mundo** |

Até hoje, todo modelo que treinamos **nasceu e morreu dentro do notebook**: a
gente treinava, dava um `print()` na acurácia... e acabou. Mas pense no
Random Forest da Aula 09: ele acerta a maioria dos passageiros. Como um
**site**, um **app de celular** ou um **chatbot** poderia usar esse modelo?

> *"Um modelo que só funciona no seu notebook é como um chef que só*
> *cozinha para si mesmo: talentoso, mas ninguém prova a comida."*

Hoje vamos transformar o nosso modelo em uma **API** — um serviço que
qualquer programa pode consultar pela rede, perguntando:
**"essa pessoa sobreviveria ao Titanic?"** 🚢

```
  ATÉ AGORA (Aulas 02–10)                  HOJE (Aula 20)
  ───────────────────────                  ──────────────────────────────────────
                                           📱 app  /  💻 site  /  🤖 chatbot
  dados → treino → modelo → print()             │  {"pclass": 3, "sex": "male", ...}
                                                ▼
                                           🌐 API Flask  ──►  🌲 modelo_titanic.pkl
                                                │
                                                ▼
                                           {"sobreviveu": 0, "probabilidade": 0.15}
```

### Roteiro de hoje

| Parte | Tema |
|-------|------|
| **1** | O que é uma API? — o garçom do restaurante |
| **2** | Congelando o modelo — treinar uma vez, salvar com `pickle` |
| **3** | Primeira API com Flask — rotas e requisições GET |
| **4** | O Oráculo do Titanic — a rota `POST /prever` (Jack vs Rose) |
| **5** | Quando o cliente erra — validação, status HTTP e levando para casa |

> **Tempo estimado: 40 minutos**

<div style="background:#d4edda; border-left:5px solid #155724; padding:14px 20px; border-radius:6px; margin:12px 0;">
<strong style="color:#155724;">Por que essa aula importa tanto:</strong>
<span style="color:#155724;"> muitos modelos de Machine Learning nunca saem do notebook de quem os criou. Saber colocar um modelo atrás de uma API é a ponte entre Ciência de Dados e Desenvolvimento de Sistemas — exatamente a sua formação em ADS. É também o que permite que um chatbot use um modelo de IA para responder perguntas.</span>
</div>

In [ ]:
import json
import os
import pickle
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd
import requests
import seaborn as sns

warnings.filterwarnings("ignore")

import flask  # já vem instalado no Colab. Rodando localmente? pip install flask
from sklearn.model_selection import train_test_split

# ── Carregando e preparando o Titanic (mesma limpeza das aulas anteriores) ────
df = sns.load_dataset("titanic").copy()

df["age"]      = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df = df.drop(columns=["deck"]).drop_duplicates().reset_index(drop=True)

df["tamanho_familia"] = df["sibsp"] + df["parch"] + 1
df["sex_enc"]         = (df["sex"] == "female").astype(int)

# As mesmas 5 features usadas desde a Aula 05
FEATURES = ["pclass", "sex_enc", "age", "tamanho_familia", "fare"]

X = df[FEATURES].copy()
y = df["survived"].copy()

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("✅ Dataset pronto!")
print(f"   Treino: {len(X_treino)} passageiros  |  Teste: {len(X_teste)} passageiros")
print(f"   Features usadas: {FEATURES}")

---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 1</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">O Que é uma API? — O Garçom do Restaurante</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Você não entra na cozinha do restaurante. Você faz o pedido ao garçom."</p>
    </div>
</div>

**API** (*Application Programming Interface*) é uma forma de um programa
**conversar com outro programa**. A melhor analogia é um restaurante:

| No restaurante 🍽️ | No mundo das APIs 🌐 |
|-------------------|----------------------|
| **Você**, o cliente | O programa que faz o pedido (app, site, **chatbot**) |
| **Cardápio** | A lista de **rotas** disponíveis e o que cada uma espera |
| **Garçom** | A **API** (Flask): recebe o pedido e traz a resposta |
| **Cozinha** | O **modelo de ML**: faz o trabalho pesado, escondido |
| **Pedido anotado** | A **requisição** (*request*), com os dados em JSON |
| **Prato servido** | A **resposta** (*response*), também em JSON |

O cliente **não precisa saber** como a cozinha funciona (se é Random Forest,
KNN ou uma rede neural). Ele só precisa saber **o que pedir** e **o que vai receber**.

### As 3 coisas que você precisa saber sobre HTTP

**1. URL = endereço + rota** — `http://127.0.0.1:5000/prever` → o restaurante
fica em `127.0.0.1:5000` (o seu próprio computador, porta 5000) e o "prato"
pedido é `/prever`.

**2. Métodos** — o *tipo* de pedido:

| Método | Quando usar | No restaurante |
|--------|-------------|----------------|
| `GET` | **Consultar** algo, sem enviar dados | "O que tem no cardápio hoje?" |
| `POST` | **Enviar dados** para serem processados | "Quero este prato, sem cebola, ponto médio" |

**3. Status code** — um número que diz como foi o pedido:

| Código | Significado | No restaurante |
|--------|-------------|----------------|
| `200` | OK, deu tudo certo | Prato servido ✅ |
| `400` | Pedido mal feito (culpa do **cliente**) | "Quero um ... hmm ... coisa" 🤷 |
| `404` | Rota não existe | "Esse prato não está no cardápio" 📋 |
| `500` | Erro interno (culpa do **servidor**) | A cozinha pegou fogo 🔥 |

### JSON: a língua comum

Um programa em Python, outro em JavaScript, outro em Java... como eles se
entendem? Todos falam **JSON** — um formato de **texto** que, para nós,
parece muito com um dicionário Python.

In [ ]:
# JSON é a "língua franca" das APIs: texto puro, que qualquer linguagem entende
passageiro = {"pclass": 3, "sex": "male", "age": 20, "sibsp": 0, "parch": 0, "fare": 7.25}

texto_json = json.dumps(passageiro)          # dicionário Python → texto JSON (para ENVIAR)
print("Tipo antes :", type(passageiro).__name__)
print("Tipo depois:", type(texto_json).__name__)
print("Conteúdo   :", texto_json)

de_volta = json.loads(texto_json)            # texto JSON → dicionário Python (ao RECEBER)
print()
print("De volta a dicionário →  sexo:", de_volta["sex"], "| idade:", de_volta["age"])

<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 1 — (a) Para cada situação, diga o método HTTP (GET ou POST) e o status code esperado: <b>(1)</b> o chatbot só quer saber se a API está no ar; <b>(2)</b> o chatbot envia os dados de um passageiro para receber a previsão; <b>(3)</b> o chatbot envia a idade como "vinte" em vez de 20. (b) Por que a comunicação usa JSON (texto) em vez de enviar o objeto Python diretamente?</span></div>

*✏️ (a) (1) `???` / `???`  —  (2) `???` / `???`  —  (3) `???` / `???`*

*✏️ (b) Usamos JSON porque: `???`*

In [ ]:
# ── GABARITO DA MISSÃO 1 (descomente para ver) ───────────────────────────────
# print("(a) (1) GET  / 200 — só consulta, não envia dados")
# print("    (2) POST / 200 — envia os dados do passageiro para processar")
# print("    (3) POST / 400 — o pedido foi mal feito: a culpa é do cliente,")
# print("        então a API deve recusar com uma mensagem explicando o erro")
# print()
# print("(b) Um objeto Python só existe na memória do programa Python. Quem chama")
# print("    a API pode ser um app em Kotlin, um site em JavaScript ou um chatbot")
# print("    em qualquer linguagem. JSON é texto simples e padronizado: todas as")
# print("    linguagens sabem ler e escrever — é a 'língua comum' entre elas.")

---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 2</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Congelando o Modelo — Treinar Uma Vez, Servir Mil Vezes</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Treinar é caro; prever é barato. A cozinha prepara a massa de manhã, não a cada pedido."</p>
    </div>
</div>

Vamos colocar em produção o **Random Forest**, campeão do placar da Aula 09.
Ele tem uma vantagem extra para hoje: **não precisa de normalização**, então
é uma peça a menos para levar para a API.

<div style="background:#fff3cd; border-left:5px solid #856404; padding:14px 20px; border-radius:6px; margin:12px 0;">
<strong style="color:#856404;">⚠️ Atenção — o bug mais comum em produção:</strong>
<span style="color:#856404;"> se fôssemos usar KNN, SVM ou Regressão Logística, teríamos que salvar também o <code>StandardScaler</code> <b>ajustado no treino</b> e aplicá-lo na API. Criar um scaler novo na API (ou esquecer dele) faz o modelo "funcionar" sem erro nenhum... mas dar previsões sem sentido.</span>
</div>

### O que é `pickle`?

O `pickle` **congela** um objeto Python (com tudo que ele aprendeu) dentro
de um arquivo. Depois, qualquer outro programa pode **descongelar** esse
arquivo e ter o modelo pronto, sem treinar de novo. É como o **save do
videogame** 🎮: você não precisa recomeçar a fase do zero.

```
  NOTEBOOK (uma vez)                         API (sempre que ligar)
  treinar → pickle.dump() ──► 💾 modelo_titanic.pkl ──► pickle.load() → prever, prever, prever...
```

Vamos salvar um **pacote** (dicionário) com o modelo **e** as informações
de que a API vai precisar — inclusive a **ordem** das features.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# ── Treinando o campeão da Aula 09 (mesmos hiperparâmetros) ──────────────────
floresta = RandomForestClassifier(n_estimators=200, max_depth=5,
                                  min_samples_leaf=5, random_state=42)
floresta.fit(X_treino, y_treino)
acuracia = accuracy_score(y_teste, floresta.predict(X_teste))
print(f"🌲 Random Forest treinado — acurácia no teste: {acuracia:.1%}")

# ── Empacotando o modelo junto com o que a API vai precisar ──────────────────
pacote = {
    "modelo": floresta,
    "features": FEATURES,          # a ORDEM das colunas importa!
    "acuracia_teste": round(acuracia, 4),
    "treinado_em": time.strftime("%Y-%m-%d %H:%M"),
}

with open("modelo_titanic.pkl", "wb") as arquivo:   # "wb" = write binary (escrever)
    pickle.dump(pacote, arquivo)

print(f"💾 Modelo salvo em 'modelo_titanic.pkl' ({os.path.getsize('modelo_titanic.pkl') / 1024:.0f} KB)")

In [ ]:
# "Descongelando" — qualquer outro programa Python poderia fazer exatamente isso
with open("modelo_titanic.pkl", "rb") as arquivo:   # "rb" = read binary (ler)
    pacote_carregado = pickle.load(arquivo)

modelo_carregado = pacote_carregado["modelo"]
iguais = (modelo_carregado.predict(X_teste) == floresta.predict(X_teste)).all()

print("Conteúdo do pacote:", list(pacote_carregado.keys()))
print("O modelo carregado prevê exatamente igual ao original?", "✅ Sim" if iguais else "❌ Não")

### Por que guardar a ordem das features?

O modelo não sabe o **nome** das colunas quando recebe só números — ele sabe
apenas que "a 1ª coluna é X, a 2ª é Y...". Veja o que acontece com a
**Rose** (do filme) quando trocamos a ordem das colunas:

In [ ]:
# Rose: 1ª classe, mulher, 17 anos, viajava com a mãe (família = 2), passagem cara
rose_preparada = pd.DataFrame([{"pclass": 1, "sex_enc": 1, "age": 17,
                                "tamanho_familia": 2, "fare": 151.55}])

ordem_errada = ["age", "fare", "pclass", "sex_enc", "tamanho_familia"]

p_certa  = floresta.predict_proba(rose_preparada[FEATURES].values)[0, 1]
p_errada = floresta.predict_proba(rose_preparada[ordem_errada].values)[0, 1]

print(f"Ordem certa  {FEATURES}")
print(f"   → chance de sobreviver: {p_certa:.0%}")
print(f"Ordem errada {ordem_errada}")
print(f"   → chance de sobreviver: {p_errada:.0%}   😱 nenhum erro apareceu!")

<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 2 — (a) Por que a API carrega o modelo do arquivo <code>.pkl</code> em vez de treinar o Random Forest de novo a cada pedido? (b) No experimento acima, o Python mostrou algum erro quando a ordem das colunas estava errada? Por que isso é perigoso em produção?</span></div>

*✏️ (a) Não treinamos a cada pedido porque: `???`*

*✏️ (b) Apareceu erro? `???` — Isso é perigoso porque: `???`*

In [ ]:
# ── GABARITO DA MISSÃO 2 (descomente para ver) ───────────────────────────────
# print("(a) Treinar é lento e caro (200 árvores!), e cada pedido levaria segundos.")
# print("    Além disso, o resultado mudaria se os dados mudassem, e a API precisaria")
# print("    ter o dataset inteiro disponível. Carregando o .pkl, o modelo já vem")
# print("    pronto: a previsão leva milissegundos e é sempre a mesma versão do modelo.")
# print()
# print("(b) Nenhum erro! O modelo recebeu 5 números e fez a conta — só que tratou")
# print("    a idade (17) como se fosse a classe, a tarifa como idade, etc. É um")
# print("    'bug silencioso': a API responde normalmente, mas a previsão é lixo.")
# print("    Por isso salvamos a lista FEATURES junto do modelo e montamos as")
# print("    colunas na API sempre nessa ordem.")

---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 3</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Primeira API com Flask — Rotas e Requisições GET</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Com poucas linhas de código, seu computador vira um servidor."</p>
    </div>
</div>

**Flask** é um *microframework* web em Python: leve, simples e muito usado
para colocar modelos de ML no ar. A anatomia de uma API Flask:

```python
app = Flask(__name__)            # 1. cria a aplicação (abre o restaurante)

@app.route("/")                  # 2. cria uma ROTA (um item do cardápio)
def home():                      # 3. função que roda quando alguém acessa a rota
    return {"mensagem": "Olá!"}  # 4. a resposta — o dicionário vira JSON sozinho

app.run(port=5000)               # 5. liga o servidor e fica esperando pedidos
```

O `@app.route(...)` é um **decorador**: uma "etiqueta" que liga um endereço
(URL) a uma função Python.

<div style="background:#e3f2fd; border-left:5px solid #1565c0; padding:14px 20px; border-radius:6px; margin:12px 0;">
<strong style="color:#1565c0;">Como rodar um servidor dentro do notebook?</strong>
<span style="color:#1565c0;"> um servidor fica rodando <b>para sempre</b>, esperando pedidos. Se rodássemos <code>app.run()</code> numa célula, o notebook travaria nela. O truque: salvar o código em um arquivo <code>app.py</code> com o comando mágico <code>%%writefile</code> e rodá-lo <b>em segundo plano</b> (como se fosse outro terminal). Assim o notebook fica livre para fazer o papel de <b>cliente</b>.</span>
</div>

In [ ]:
%%writefile app.py
from flask import Flask

app = Flask(__name__)


@app.route("/")
def home():
    return {"mensagem": "🚢 Bem-vindo ao Oráculo do Titanic!"}


@app.route("/saudacao/<nome>")
def saudacao(nome):
    return {"mensagem": f"Olá, {nome}! Pronto para embarcar?"}


if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5000)

In [ ]:
# ── Ferramentas para ligar/desligar o servidor em segundo plano ──────────────
# (não precisa decorar este código — é só a "chave de ignição" do servidor)
URL = "http://127.0.0.1:5000"
servidor = globals().get("servidor")   # lembra de um servidor que já estava ligado


def desligar_servidor():
    global servidor
    if servidor is not None:
        servidor.terminate()
        servidor.wait()
        servidor = None
        print("⚪ Servidor desligado")


def ligar_servidor(arquivo="app.py"):
    """Roda o arquivo em segundo plano (como em outro terminal) e espera a API responder."""
    global servidor
    desligar_servidor()                          # se já havia um ligado, desliga antes
    log = open("servidor.log", "w")
    servidor = subprocess.Popen([sys.executable, arquivo], stdout=log, stderr=subprocess.STDOUT)
    for _ in range(40):                          # espera até ~10 segundos
        if servidor.poll() is not None:          # o processo morreu? então deu erro
            break
        try:
            if requests.get(URL, timeout=0.5).ok:
                print(f"🟢 Servidor no ar em {URL}")
                return
        except requests.exceptions.ConnectionError:
            pass
        time.sleep(0.25)
    print("🔴 O servidor não subiu. Mensagem de erro:\n")
    print(open("servidor.log", encoding="utf-8", errors="replace").read())


ligar_servidor()

Servidor ligado! Agora o notebook vira o **cliente**. A biblioteca
`requests` faz o papel do navegador (ou do app, ou do chatbot):

In [ ]:
# Pedido 1: GET na rota principal
resposta = requests.get(f"{URL}/")
print("GET /                →", resposta.status_code, resposta.json())

# Pedido 2: rota com parâmetro na própria URL
resposta = requests.get(f"{URL}/saudacao/Rose")
print("GET /saudacao/Rose   →", resposta.status_code, resposta.json())

# Pedido 3: uma rota que NÃO existe no cardápio
resposta = requests.get(f"{URL}/cozinha")
print("GET /cozinha         →", resposta.status_code, "(sem JSON: página de erro do Flask)")

> 💡 **Rodando no seu computador (Jupyter/VS Code)?** Com o servidor ligado,
> abra http://127.0.0.1:5000/saudacao/SeuNome no navegador! No **Colab** isso
> não funciona: o servidor está rodando numa máquina do Google, não no seu PC —
> por isso usamos o `requests` de dentro do próprio notebook.

<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 3 — (a) Adicione ao <code>app.py</code> (na célula do <code>%%writefile</code>) uma nova rota GET <code>/tripulacao</code> que devolva <code>{"capitao": "Edward Smith", "navio": "RMS Titanic"}</code>. Rode a célula de novo, depois rode <code>ligar_servidor()</code> e teste com a célula abaixo. (b) Antes de criar a rota, qual status code a célula de teste devolve? E depois? Por quê?</span></div>

In [ ]:
# ✏️ Teste da Missão 3 — rode ANTES e DEPOIS de criar a rota
ligar_servidor()   # religa o servidor para ele ler o app.py atualizado
resposta = requests.get(f"{URL}/tripulacao")
print("Status:", resposta.status_code)
print("Resposta:", resposta.json() if resposta.ok else resposta.text[:80])

*✏️ (b) Antes: `???` — Depois: `???` — porque: `???`*

In [ ]:
# ── GABARITO DA MISSÃO 3 (descomente para ver) ───────────────────────────────
# print("(a) Basta acrescentar no app.py, antes do 'if __name__ == ...':")
# print("""
# @app.route("/tripulacao")
# def tripulacao():
#     return {"capitao": "Edward Smith", "navio": "RMS Titanic"}
# """)
# print("(b) Antes: 404 (Not Found) — a rota não estava no 'cardápio' da API.")
# print("    Depois: 200 (OK) — a rota existe e respondeu com o JSON.")
# print("    Lembre: o servidor precisa ser RELIGADO para ler o arquivo novo!")

---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 4</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">O Oráculo do Titanic — A Rota POST /prever</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Agora o garçom leva o pedido até a cozinha."</p>
    </div>
</div>

Hora de colocar o modelo atrás da API. O caminho de um pedido:

```
 o cliente envia             a API Flask faz                              e devolve
 ───────────────             ──────────────────────────────────────       ─────────────────────
 {"pclass": 3,          ┌──► 1. lê o JSON           (request.get_json)
  "sex": "male",        │    2. valida os campos    (faltou algo? → 400)
  "age": 20,       ─────┤    3. REPETE A PREPARAÇÃO DO TREINO:            {"sobreviveu": 0,
  "sibsp": 0,           │       sex → sex_enc                       ───►   "probabilidade_...": 0.15,
  "parch": 0,           │       sibsp + parch + 1 → tamanho_familia        "mensagem": "🧊 ..."}
  "fare": 7.75}         │    4. monta as colunas na ordem FEATURES
                        └──► 5. modelo.predict_proba()
```

<div style="background:#d1ecf1; border-left:5px solid #0c5460; padding:14px 20px; border-radius:6px; margin:12px 0;">
<strong style="color:#0c5460;">A ideia mais importante da aula:</strong>
<span style="color:#0c5460;"> o cliente envia dados <b>"crus"</b>, do jeito que um usuário conhece (sexo, irmãos, pais). Ele não sabe o que é <code>sex_enc</code> ou <code>tamanho_familia</code> — e nem deveria precisar saber! A API é que precisa aplicar <b>exatamente a mesma preparação</b> que fizemos no treino (Aula 03). Por isso ela tem a função <code>preparar_passageiro()</code>.</span>
</div>

Outros detalhes novos no código abaixo:

- `methods=["POST"]` → essa rota aceita **receber dados**
- `request.get_json()` → lê o JSON que o cliente enviou e transforma em dicionário
- `return jsonify(...), 400` → devolve uma resposta **com um status code** específico
- O modelo é carregado **uma única vez**, quando o servidor liga — não a cada pedido

In [ ]:
%%writefile app.py
import os
import pickle

import pandas as pd
from flask import Flask, jsonify, request

app = Flask(__name__)

# ── 1. Carrega o modelo UMA VEZ, quando o servidor liga (não a cada pedido!) ──
PASTA = os.path.dirname(os.path.abspath(__file__))
with open(os.path.join(PASTA, "modelo_titanic.pkl"), "rb") as arquivo:
    pacote = pickle.load(arquivo)

modelo = pacote["modelo"]
FEATURES = pacote["features"]

# Campos que o cliente precisa enviar (dados "crus", como o usuário conhece)
CAMPOS_OBRIGATORIOS = ["pclass", "sex", "age", "sibsp", "parch", "fare"]


def preparar_passageiro(dados):
    """Repete na API exatamente a mesma preparação feita no treino."""
    passageiro = {
        "pclass": int(dados["pclass"]),
        "sex_enc": 1 if dados["sex"] == "female" else 0,
        "age": float(dados["age"]),
        "tamanho_familia": int(dados["sibsp"]) + int(dados["parch"]) + 1,
        "fare": float(dados["fare"]),
    }
    return pd.DataFrame([passageiro])[FEATURES]   # mesma ordem de colunas do treino


@app.route("/")
def home():
    return jsonify({"mensagem": "🚢 Bem-vindo ao Oráculo do Titanic! Use POST /prever para consultar."})


@app.route("/info")
def info():
    return jsonify({
        "modelo": type(modelo).__name__,
        "acuracia_teste": pacote["acuracia_teste"],
        "treinado_em": pacote["treinado_em"],
        "campos_esperados": CAMPOS_OBRIGATORIOS,
    })


@app.route("/prever", methods=["POST"])
def prever():
    dados = request.get_json(silent=True)

    # ── 2. Validação: o pedido veio completo? ──
    if dados is None:
        return jsonify({"erro": "Envie os dados do passageiro em formato JSON."}), 400

    faltando = [campo for campo in CAMPOS_OBRIGATORIOS if campo not in dados]
    if faltando:
        return jsonify({"erro": f"Campos obrigatórios faltando: {faltando}"}), 400

    try:
        X = preparar_passageiro(dados)
    except (ValueError, TypeError):
        return jsonify({"erro": "pclass, age, sibsp, parch e fare precisam ser números."}), 400

    # ── 3. Previsão ──
    sobreviveu = int(modelo.predict(X)[0])
    probabilidade = float(modelo.predict_proba(X)[0, 1])

    return jsonify({
        "sobreviveu": sobreviveu,
        "probabilidade_sobrevivencia": round(probabilidade, 4),
        "mensagem": "🛟 Conseguiu um lugar no bote!" if sobreviveu else "🧊 Não resistiu ao naufrágio...",
    })


if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5000)

In [ ]:
ligar_servidor()   # religa o servidor com a nova versão do app.py

# A rota /info funciona como o "cardápio": diz o que a API espera receber
print(json.dumps(requests.get(f"{URL}/info").json(), indent=2, ensure_ascii=False))

### 🎬 Jack vs Rose

Vamos consultar o Oráculo sobre os dois protagonistas do filme *Titanic* (1997):

| | Jack Dawson 🃏 | Rose DeWitt Bukater 💎 |
|---|---|---|
| Classe | 3ª (ganhou a passagem no pôquer) | 1ª |
| Sexo | masculino | feminino |
| Idade | 20 | 17 |
| A bordo com ele(a) | ninguém | a mãe (`parch = 1`) |
| Tarifa | barata | cara |

Repare no `requests.post(..., json=passageiro)`: o parâmetro `json=` já faz
o `json.dumps()` e avisa ao servidor que o conteúdo é JSON. Prático!

In [ ]:
jack = {"pclass": 3, "sex": "male",   "age": 20, "sibsp": 0, "parch": 0, "fare": 7.75}
rose = {"pclass": 1, "sex": "female", "age": 17, "sibsp": 0, "parch": 1, "fare": 151.55}

for nome, passageiro in [("Jack", jack), ("Rose", rose)]:
    resposta = requests.post(f"{URL}/prever", json=passageiro)
    r = resposta.json()
    print(f"{nome:<5} [{resposta.status_code}] → {r['mensagem']}  "
          f"(probabilidade: {r['probabilidade_sobrevivencia']:.0%})")

print()
print("Resposta completa da Rose (o que um app ou chatbot receberia):")
print(json.dumps(r, indent=2, ensure_ascii=False))

<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 4 — <b>E você, sobreviveria?</b> (a) Preencha <code>meu_passageiro</code> com os seus dados (ou de um personagem inventado) e consulte a API. Qual foi a sua probabilidade? (b) Rode o experimento seguinte, que muda <b>só a classe</b> (e a tarifa típica dela). Quanto a probabilidade muda da 1ª para a 3ª classe? O que isso diz sobre o Titanic?</span></div>

In [ ]:
# ✏️ MISSÃO 4 (a) — troque pelos SEUS dados
meu_passageiro = {
    "pclass": 2,        # 1 = primeira, 2 = segunda, 3 = terceira classe
    "sex": "female",    # "male" ou "female"
    "age": 21,
    "sibsp": 0,         # nº de irmãos/cônjuge a bordo
    "parch": 0,         # nº de pais/filhos a bordo
    "fare": 19.0,       # tarifa típica: 1ª ~ 62 | 2ª ~ 19 | 3ª ~ 8
}

r = requests.post(f"{URL}/prever", json=meu_passageiro).json()
print(f"🔮 O Oráculo diz: {r['mensagem']}  (probabilidade: {r['probabilidade_sobrevivencia']:.0%})")

In [ ]:
# MISSÃO 4 (b) — mesmo passageiro, só mudando a classe (com a tarifa mediana de cada uma)
tarifa_tipica = df.groupby("pclass")["fare"].median()

for classe in [1, 2, 3]:
    teste = {**meu_passageiro, "pclass": classe, "fare": float(tarifa_tipica[classe])}
    r = requests.post(f"{URL}/prever", json=teste).json()
    barra = "█" * round(r["probabilidade_sobrevivencia"] * 30)
    print(f"  {classe}ª classe (tarifa {tarifa_tipica[classe]:>5.1f}) → "
          f"{r['probabilidade_sobrevivencia']:>4.0%} {barra}")

*✏️ (a) Minha probabilidade de sobreviver foi: `???`*

*✏️ (b) Da 1ª para a 3ª classe a probabilidade mudou de `???` para `???`. Isso mostra que: `???`*

In [ ]:
# ── GABARITO DA MISSÃO 4 (descomente para ver) ───────────────────────────────
# print("(a) Resposta pessoal — cada um terá um resultado diferente!")
# print()
# print("(b) A probabilidade cai bastante da 1ª para a 3ª classe, mesmo mantendo")
# print("    sexo, idade e família iguais. O modelo aprendeu com os dados reais")
# print("    que a classe social pesou na sobrevivência: os passageiros da 1ª")
# print("    classe estavam mais perto dos botes e tiveram prioridade no embarque.")
# print("    O que vimos na EDA da Aula 02 agora está 'embutido' na API.")

---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 5</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Quando o Cliente Erra — Validação e Status HTTP</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Em produção, o usuário SEMPRE vai mandar algo que você não esperava."</p>
    </div>
</div>

Uma API pública recebe pedidos de qualquer lugar. Ela precisa **se defender**
de dados errados e responder com um **erro claro** (status `400`), em vez de
quebrar (status `500`) ou — pior ainda — responder algo errado sem avisar.

Vamos fazer alguns pedidos "ruins" e ver como a nossa API reage:

In [ ]:
pedidos_ruins = {
    "Esqueceu a idade":  {"pclass": 3, "sex": "male", "sibsp": 0, "parch": 0, "fare": 7.25},
    "Idade por extenso": {"pclass": 3, "sex": "male", "age": "vinte", "sibsp": 0, "parch": 0, "fare": 7.25},
    "Pedido vazio":      None,
}

for descricao, corpo in pedidos_ruins.items():
    resposta = requests.post(f"{URL}/prever", json=corpo)
    print(f"{descricao:<18} → [{resposta.status_code}] {resposta.json()}")

A API recusou os três pedidos com status `400` e uma mensagem explicando o
problema. 👏 Mas existe um erro mais traiçoeiro... Veja o que acontece se o
app de alguém mandar a Rose com `"Female"` (com F maiúsculo):

In [ ]:
rose_maiuscula = {**rose, "sex": "Female"}   # mesma Rose, só com F maiúsculo

for descricao, passageiro in [('"female"', rose), ('"Female"', rose_maiuscula)]:
    resposta = requests.post(f"{URL}/prever", json=passageiro)
    r = resposta.json()
    print(f"Rose com sex={descricao:<9} → [{resposta.status_code}] {r['mensagem']}  "
          f"(probabilidade: {r['probabilidade_sobrevivencia']:.0%})")

<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 5 (final) — (a) A API respondeu <code>200 OK</code> para a Rose com <code>"Female"</code>, mas a previsão mudou. Olhando a função <code>preparar_passageiro()</code>, explique por quê. (b) Corrija o <code>app.py</code> da Parte 4: antes do <code>try</code>, converta o sexo para minúsculas e devolva erro <code>400</code> se ele não for <code>"male"</code> nem <code>"female"</code>. Rode a célula do <code>%%writefile</code> de novo e teste com a célula abaixo.</span></div>

In [ ]:
# ✏️ Teste da Missão 5 — depois da correção, "Female" deve funcionar e "mulher" deve dar 400
ligar_servidor()
for valor in ["female", "Female", "FEMALE", "mulher"]:
    resposta = requests.post(f"{URL}/prever", json={**rose, "sex": valor})
    print(f"sex={valor!r:<9} → [{resposta.status_code}] {resposta.json()}")

*✏️ (a) A previsão mudou porque: `???`*

*✏️ (b) Código que adicionei no `app.py`: `???`*

In [ ]:
# ── GABARITO DA MISSÃO 5 (descomente para ver) ───────────────────────────────
# print("(a) Em preparar_passageiro() temos: 1 if dados['sex'] == 'female' else 0")
# print("    Como 'Female' != 'female', a comparação dá False e a Rose vira")
# print("    sex_enc = 0 (homem)! Nenhum erro aparece: é um bug silencioso,")
# print("    o tipo mais perigoso — o cliente confia numa resposta errada.")
# print()
# print("(b) Dentro de prever(), logo antes do 'try:', acrescente:")
# print("""
#     dados["sex"] = str(dados["sex"]).strip().lower()
#     if dados["sex"] not in ("male", "female"):
#         return jsonify({"erro": "O campo 'sex' precisa ser 'male' ou 'female'."}), 400
# """)
# print("    Essa é exatamente a versão do app.py que está na pasta")
# print("    Aula_20_API_Flask_Titanic/ do repositório.")

---

## 🏠 Levando para casa — rodando fora do notebook

No mundo real, a API não roda dentro de um notebook. Na pasta
[`Aula_20_API_Flask_Titanic/`](https://github.com/FIAPON/fiap-graduacao-iachatbot/tree/main/Aula_20_API_Flask_Titanic)
do repositório estão os mesmos códigos desta aula, separados em arquivos:

| Arquivo | Papel |
|---------|-------|
| `treinar_modelo.py` | Treina o Random Forest e salva o `modelo_titanic.pkl` (Parte 2) |
| `app.py` | A API Flask completa, já com a Missão 5 resolvida (Partes 4 e 5) |
| `cliente.py` | Um **mini-chatbot de terminal** que pergunta seus dados e consulta a API |

```bash
pip install -r requirements.txt
python treinar_modelo.py   # 1. gera o modelo (uma vez só)
python app.py              # 2. TERMINAL 1: liga o servidor e deixa aberto
python cliente.py          # 3. TERMINAL 2: conversa com o Oráculo
```

### E o deploy "de verdade"?

O `app.run()` liga um **servidor de desenvolvimento** — ótimo para aprender,
mas não para milhares de usuários. Em produção, a mesma API roda com um
servidor robusto (como `gunicorn` ou `waitress`) em um serviço de nuvem
(Render, Railway, AWS, Azure, Google Cloud...), que dá a ela um **endereço
público** na internet, como `https://oraculo-titanic.onrender.com/prever`.
O código Flask continua praticamente o mesmo!

### E o chatbot? 🤖

É aqui que as duas metades da disciplina se encontram. Um chatbot pode
**conversar** com o usuário, coletar as informações e **chamar a nossa API**
por trás dos panos:

```
👤 "Oi! Eu sobreviveria ao Titanic?"
🤖 "Vamos descobrir! Em qual classe você viajaria?"   ← coleta os dados na conversa
👤 "Na terceira, tenho 20 anos, viajaria sozinho"
         │
         └──► POST /prever {"pclass": 3, "sex": "male", "age": 20, ...}
                  ◄── {"sobreviveu": 0, "probabilidade_sobrevivencia": 0.15}
🤖 "Sinto muito... você teria cerca de 15% de chance. Que tal a 1ª classe? 😅"
```

O `cliente.py` da pasta é exatamente uma versão simples disso!

In [ ]:
# Ao terminar a aula, desligue o servidor para liberar a porta 5000
desligar_servidor()

**✏️ Minha reflexão sobre a aula:**

1. API em uma frase simples: *...*

2. A API precisa repetir a preparação dos dados do treino porque: *...*

3. Um sistema que eu gostaria de construir consumindo uma API de modelo de ML: *...*